In [9]:
import math
import numpy as np


def t_to_idx(t: np.ndarray, T: int) -> np.ndarray:
    return np.round(t * (T - 1)).astype(np.int64)


def est_directional_similarity(xs: np.ndarray, n_est: int = 1000) -> np.ndarray:
    """xs: (batch, nx). Returns (n_est, ) between 0 and 1."""
    batch, nx = xs.shape

    # Center first
    xs = xs - np.mean(xs, axis=0, keepdims=True)

    rand_idxs1 = np.random.randint(batch, size=n_est)
    rand_idxs2 = np.random.randint(batch, size=n_est)

    xs1 = xs[rand_idxs1]
    xs2 = xs[rand_idxs2]

    # Normalize to unit vector
    xs1 /= np.linalg.norm(xs1, axis=1, keepdims=True)
    xs2 /= np.linalg.norm(xs2, axis=1, keepdims=True)

    cos_angle = np.sum(xs1 * xs2, axis=1).clip(-1.0, 1.0)
    angle = np.arccos(cos_angle)

    D_ij = 1.0 - angle / np.pi
    return D_ij


def opinion_thresh(inner: np.ndarray) -> np.ndarray:
    return 2.0 * (inner > 0) - 1.0


def compute_mean_drift_term(mf_x: np.ndarray, xi: np.ndarray) -> np.ndarray:
    """Compute the mean drift term B(p,ξ)"""
    B, *Ts, D = mf_x.shape
    assert xi.shape == (*Ts, D)

    mf_x_norm = np.linalg.norm(mf_x, axis=-1, keepdims=True)
    assert np.all(mf_x_norm > 0.0)

    normalized_mf_x = mf_x / np.sqrt(mf_x_norm)

    mf_agree_j = opinion_thresh(np.sum(mf_x * xi, axis=-1, keepdims=True))
    mean_drift_term = np.mean(mf_agree_j * normalized_mf_x, axis=0)

    mean_drift_term_norm = np.linalg.norm(mean_drift_term, axis=-1, keepdims=True)
    mean_drift_term = mean_drift_term / np.sqrt(mean_drift_term_norm)
    
    return mean_drift_term


def opinion_f(x: np.ndarray, mf_drift: np.ndarray, xi: np.ndarray) -> np.ndarray:
    """Compute the polarize dynamic"""
    b, T, nx = x.shape
    assert xi.shape == mf_drift.shape == (T, nx)

    agree_i = opinion_thresh(np.sum(x * xi, axis=-1, keepdims=True))
    agree_i[agree_i == 0] = 1.0

    abs_sqrt_agree_i = np.sqrt(np.abs(agree_i))
    assert np.all(abs_sqrt_agree_i > 0.0)

    norm_agree_i = agree_i / abs_sqrt_agree_i
    f = norm_agree_i * mf_drift
    
    return f


def build_f_mul(T, coeff=8.0) -> np.ndarray:
    ts = np.linspace(0.0, 1.0, T)
    f_mul = np.clip(1.0 - np.exp(coeff * (ts - 1.0)) + 1e-5, 1e-4, 1.0)
    f_mul = f_mul**5.0
    return f_mul


def build_xis(T, D) -> np.ndarray:
    rng = np.random.default_rng(seed=4078213)
    xis = rng.standard_normal((T, D))

    # Construct Brownian motion xis
    xi = xis[0]
    bm_xis = [xi]
    std = 0.4
    dt = 1.0 / T
    for t in range(1, T):
        xi = xi - (2.0 * xi) * dt + std * np.sqrt(dt) * xis[t]
        bm_xis.append(xi)

    xis = np.stack(bm_xis)
    xis /= np.linalg.norm(xis, axis=-1, keepdims=True)

    print("USING BM XI! xis.sum(): {}".format(np.sum(xis)))
    assert xis.shape == (T, D)
    return xis


def proj_pca(xs_f: np.ndarray):
    """Project to PCA space"""
    batch, T, nx = xs_f.shape
    flat_xsf = xs_f.reshape(-1, *xs_f.shape[2:])

    # Use final timestep for PCA
    final_xs_f = xs_f[:, -1, :]
    mean_pca_xs = np.mean(final_xs_f, axis=0, keepdims=True)
    final_xs_f -= mean_pca_xs

    if batch > 200:
        rand_idxs = np.random.permutation(batch)[:200]
        final_xs_f = final_xs_f[rand_idxs]

    # Perform SVD
    U, S, VT = np.linalg.svd(final_xs_f, full_matrices=False)
    VT = VT[:2, :]  # Keep first two components
    
    V = VT.T
    flat_xsf -= mean_pca_xs
    proj_xs_f = flat_xsf @ V
    proj_xs_f = proj_xs_f.reshape(batch, T, *proj_xs_f.shape[1:])
    
    return proj_xs_f, V


class PolarizeDyn:
    def __init__(self, D) -> None:
        self.S = 500
        self.D = D
        self.polarize_strength = 6.0
        self.xis = build_xis(self.S, self.D)
        self.f_muls = build_f_mul(self.S, coeff=8.0)
        self.mf_drift = np.zeros((self.S, self.D))
        self.is_mf_drift_set = False

    def set_mf_drift(self, mf_xs):
        t = np.arange(mf_xs.shape[1]) / mf_xs.shape[1]
        t_idx = t_to_idx(t, self.S)
        xi = self.xis[t_idx]
        # assert mf_xs.shape[1:] == (self.S, self.D)
        mf_drift = compute_mean_drift_term(mf_xs, xi)
        # assert mf_drift.shape == (self.S, self.D)
        self.mf_drift = mf_drift
        self.is_mf_drift_set = True

    def __call__(self, xs, t):
        t = t.reshape(-1)
        (T, D) = xs.shape[0], xs.shape[1]
        assert t.shape == (T,)

        xs = xs.reshape(1, T, D)
        t_idx = t_to_idx(t, self.S)
        fmul = self.f_muls[t_idx]
        
        xi = self.xis[t_idx]
        mf_drift = self.mf_drift[t_idx]
            
        f = self.polarize_strength * opinion_f(xs, mf_drift, xi)
        f = fmul.reshape(1, -1, 1) * f
        
        return f.reshape(T, D)

In [15]:
pol = PolarizeDyn(D=1000)
pol.set_mf_drift(np.ones((128, 30, 1000)))

USING BM XI! xis.sum(): 203.87683298660696


In [16]:
pol(np.random.normal(1, 1, size=(50, 1000)),  np.linspace(0, 0.9, 50) ).mean()

IndexError: index 64 is out of bounds for axis 0 with size 60